# Notebook 02 — Calidad, limpieza y preparación

**Objetivo:** Preparar el dataset a partir de decisiones justificadas con evidencia observada en la inspección inicial. Cada decisión indica la evidencia que la motivó, la acción aplicada y el impacto observado.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

with open('../data/raw/streaming_users_dirty.json') as f:
    data = json.load(f)
df = pd.DataFrame(data)
print(f'Dataset cargado. Dimensiones iniciales: {df.shape}')

## Paso 1 — Eliminación de filas duplicadas

In [ ]:
# Evidencia: la inspección detectó 126 filas completamente duplicadas.
# Representan registros repetidos que distorsionarían cualquier agregación o análisis.
# Decisión: eliminar duplicados conservando la primera ocurrencia.
n_antes = len(df)
df = df.drop_duplicates()
print(f'Filas eliminadas: {n_antes - len(df)} | Filas restantes: {len(df)}')

## Paso 2 — Normalización de subscription_plan

In [ ]:
# Evidencia: 15 variantes para 3 planes reales. Sin normalización, los análisis
# agrupados por plan producirían resultados fragmentados e incorrectos.
print('Antes:', df['subscription_plan'].value_counts().to_dict())

plan_map = {
    'Std': 'Estándar', 'estandar': 'Estándar', 'Estándar ': 'Estándar', 'STANDARD': 'Estándar',
    'basico': 'Básico', 'básico': 'Básico', 'BASICO': 'Básico', 'Basic': 'Básico',
    'premium': 'Premium', 'Premium ': 'Premium', 'Premiun': 'Premium', 'PREMIUM': 'Premium'
}
df['subscription_plan'] = df['subscription_plan'].replace(plan_map)
print('Después:', df['subscription_plan'].value_counts().to_dict())

## Paso 3 — Normalización de country

In [ ]:
# Evidencia: 26 variantes para 7 países. Mezcla de idiomas (Brazil/Brasil),
# capitalización (brasil/Brasil) y códigos ISO (BRA, MEX, ARG, COL, URY, PER, CHL).
country_map = {
    'Brazil': 'Brasil', 'brasil': 'Brasil', 'BRA': 'Brasil',
    'méxico': 'México', 'Mexico': 'México', 'MEX': 'México',
    'chile': 'Chile', 'Chile ': 'Chile', 'CHL': 'Chile',
    'ARG': 'Argentina', 'argentina': 'Argentina', 'Argentina ': 'Argentina',
    'COL': 'Colombia', 'colombia': 'Colombia',
    'URY': 'Uruguay', 'uruguay': 'Uruguay',
    'PER': 'Perú', 'Peru': 'Perú', 'perú': 'Perú'
}
df['country'] = df['country'].replace(country_map)
print('Países resultantes:', df['country'].unique())

## Paso 4 — Normalización de favorite_genre

In [ ]:
# Evidencia: 29 variantes para 7 géneros. Mezcla de idiomas (comedy/Comedia,
# Documentary/Documental, Action/Acción), capitalización y errores tipográficos (thriler).
genre_map = {
    'CRIME': 'Crime', 'Crimen': 'Crime', 'crime': 'Crime',
    'THRILLER': 'Thriller', 'thriler': 'Thriller', 'Thriller ': 'Thriller',
    'DRAMA': 'Drama', 'drama': 'Drama', 'Drama ': 'Drama',
    'ACCIÓN': 'Acción', 'accion': 'Acción', 'Action': 'Acción',
    'ROMANCE': 'Romance', 'romance': 'Romance', 'Romance ': 'Romance',
    'COMEDIA': 'Comedia', 'comedy': 'Comedia', 'Comedia ': 'Comedia',
    'Documental': 'Documental', 'Documentary': 'Documental',
    'DOC': 'Documental', 'documental': 'Documental'
}
df['favorite_genre'] = df['favorite_genre'].replace(genre_map)
print('Géneros resultantes:', df['favorite_genre'].dropna().unique())

## Paso 5 — Eliminación de outliers en age

In [ ]:
# Evidencia: valores -5 (imposible), 130 y 150 (biológicamente inverosímiles).
# No pueden imputarse porque son errores de carga, no ausencias de información.
# Se considera rango válido: 13-100 años (edad mínima razonable para suscripción).
n = len(df)
print('Distribución de valores extremos en age:')
print(df[df['age'] < 13]['age'].value_counts())
print(df[df['age'] > 100]['age'].value_counts())
df = df[(df['age'] >= 13) & (df['age'] <= 100)]
print(f'Eliminados: {n - len(df)} | Restantes: {len(df)}')

## Paso 6 — Eliminación de outliers en monthly_watch_time_mins

In [ ]:
# Evidencia: valores negativos (-120, -1) son físicamente imposibles para minutos de
# visualización. Valores 50000 y 99999 equivalen a 833+ horas mensuales, lo que supera
# las horas disponibles en un mes (720h), siendo errores de carga evidentes.
# Umbral máximo: 10000 minutos (~167 horas/mes, margen amplio pero razonable).
n = len(df)
df = df[(df['monthly_watch_time_mins'].isna()) | 
        ((df['monthly_watch_time_mins'] >= 0) & (df['monthly_watch_time_mins'] <= 10000))]
print(f'Eliminados: {n - len(df)} | Restantes: {len(df)}')

## Paso 7 — Eliminación de outliers en customer_support_tickets

In [ ]:
# Evidencia: valor -1 es imposible (no puede haber tickets negativos).
# Valores 99 y 150 son inverosímiles para tickets mensuales de soporte de un
# usuario individual. Se establece umbral máximo de 50.
n = len(df)
df = df[(df['customer_support_tickets'] >= 0) & (df['customer_support_tickets'] <= 50)]
print(f'Eliminados: {n - len(df)} | Restantes: {len(df)}')

## Paso 8 — Imputación de nulos en monthly_watch_time_mins

In [ ]:
# Evidencia: quedan nulos tras eliminar outliers. La distribución del tiempo de
# visualización es aproximadamente simétrica, por lo que la mediana es un estimador robusto.
mediana = df['monthly_watch_time_mins'].median()
print(f'Nulos antes: {df["monthly_watch_time_mins"].isnull().sum()}')
print(f'Mediana utilizada: {mediana:.1f} minutos')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df['monthly_watch_time_mins'].dropna().hist(ax=axes[0], bins=30, color='steelblue')
axes[0].set_title('Distribución antes de imputar')
axes[0].axvline(mediana, color='red', linestyle='--', label=f'Mediana: {mediana:.0f}')
axes[0].legend()
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].fillna(mediana)
df['monthly_watch_time_mins'].hist(ax=axes[1], bins=30, color='steelblue')
axes[1].set_title('Distribución tras imputar')
plt.tight_layout()
plt.show()
print(f'Nulos después: {df["monthly_watch_time_mins"].isnull().sum()}')

## Paso 9 — Imputación de nulos en favorite_genre

In [ ]:
# Evidencia: variable categórica nominal sin orden natural. No es posible aplicar mediana.
# Se imputa con la moda (género más frecuente), que es la alternativa más conservadora
# para variables categóricas cuando los nulos no representan una categoría propia.
moda = df['favorite_genre'].mode()[0]
print(f'Nulos antes: {df["favorite_genre"].isnull().sum()}')
print(f'Moda utilizada: {moda}')
df['favorite_genre'] = df['favorite_genre'].fillna(moda)
print(f'Nulos después: {df["favorite_genre"].isnull().sum()}')

## Paso 10 — Corrección de last_login_date

In [ ]:
# Evidencia: formatos mixtos (YYYY-MM-DD y YYYY/MM/DD) y fechas futuras imposibles.
# Acción 1: unificar formatos con pd.to_datetime(format='mixed').
# Acción 2: eliminar registros con fecha posterior a hoy (2025-06-27).
# Acción 3: imputar nulos restantes con la mediana de fechas válidas.
df['last_login_date'] = pd.to_datetime(df['last_login_date'], format='mixed', errors='coerce')
n = len(df)
df = df[df['last_login_date'] <= pd.Timestamp('2025-06-27')]
print(f'Registros con fecha futura eliminados: {n - len(df)}')
mediana_fecha = df['last_login_date'].median()
df['last_login_date'] = df['last_login_date'].fillna(mediana_fecha)
print(f'Nulos en fecha tras imputar: {df["last_login_date"].isnull().sum()}')

## Paso 11 — Verificación final y guardado

In [ ]:
print('=== Estado final del dataset ===')
print(f'Dimensiones: {df.shape}')
print(f'Retención: {len(df)/8160*100:.1f}% de los registros originales')
print()
print('Nulos restantes:')
print(df.isnull().sum())
print()
df.describe()

In [ ]:
df.to_csv('../data/processed/streaming_users_clean.csv', index=False)
print('Dataset procesado guardado en data/processed/streaming_users_clean.csv')